In [ ]:
import os
import hashlib
import json

import pandas as pd
import urllib.request

## **Descarga de datos**

In [ ]:
def download_data(url, filename):
    """Downloads data from a URL and saves it to a file."""
    print(f"Downloading {url} to {filename}...")
    try:
        urllib.request.urlretrieve(url, filename)
        print(f" Downloaded {filename}")
    except Exception as e:
        print(f" Error downloading {filename}: {e}")
        raise

Datos de duración de viajes entre puntos en NYC

In [ ]:
jan_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-01.parquet"
feb_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-02.parquet"

In [ ]:
output_path = "data/processed"
data_path = "data"

In [ ]:
os.makedirs(output_path, exist_ok=True)

In [ ]:
download_data(jan_url, os.path.join(data_path, "jan.parquet"))
download_data(feb_url, os.path.join(data_path, "feb.parquet"))

## **Carga de datos en pandas**

In [ ]:
df_jan = pd.read_parquet(os.path.join(data_path, "jan.parquet"))
df_feb = pd.read_parquet(os.path.join(data_path, "feb.parquet"))

In [ ]:
df_jan.head()

In [ ]:
def duration_trip(df):
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    return df

In [ ]:
df_jan = duration_trip(df_jan)
df_feb = duration_trip(df_feb)

In [ ]:
df_jan.head()

In [ ]:
df_jan.columns

In [ ]:
df_jan.trip_type.unique()

In [ ]:
df_jan.drop(columns=['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'VendorID'], inplace=True)
df_feb.drop(columns=['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'VendorID'], inplace=True)

In [ ]:
# save processed data
df_jan.to_parquet("data/processed/jan.parquet")
df_feb.to_parquet("data/processed/feb.parquet")

## **Metadata del dataset ("dataset version")**

En MLOps es importante poder identificar qué versión de datos usaste.

Aquí guardaremos un `metadata.json` simple con:
- cantidad de filas/columnas
- nombres de columnas
- checksum (hash) de los parquet procesados

In [ ]:
def file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

metadata = {
    "jan": {
        "rows": int(df_jan.shape[0]),
        "cols": int(df_jan.shape[1]),
        "columns": list(df_jan.columns),
        "path": "data/processed/jan.parquet",
        "sha256": file_sha256("data/processed/jan.parquet"),
    },
    "feb": {
        "rows": int(df_feb.shape[0]),
        "cols": int(df_feb.shape[1]),
        "columns": list(df_feb.columns),
        "path": "data/processed/feb.parquet",
        "sha256": file_sha256("data/processed/feb.parquet"),
    },
}

metadata_path = "data/processed/metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

metadata